# Gold Date Dimension (`gold_dim_date`)

## Notebook Overview

### Purpose
Build the Gold date dimension from the transaction dates available in the Silver layer. This dimension provides a consistent calendar structure for time-based sales analysis, reporting, and dashboard development.

### Source
- **Input table:** Silver Transactions
- **Input path:** `silver/transactions`
- **Source column:** `transaction_date`

### Transformation Logic
1. Read transaction dates from the Silver transactions dataset.
2. Identify the minimum and maximum transaction dates.
3. Generate a continuous calendar covering the entire date range, including dates with no transactions.
4. Derive calendar attributes such as date key, day, month, month name, quarter, year, and day of week.
5. Add `_gold_processed_timestamp` for auditability.
6. Validate the generated dimension before writing it to the Gold layer.

### Target
- **Gold dimension:** `gold_dim_date`
- **Output path:** `gold/dim_date`
- **Storage format:** Parquet
- **Grain:** One row per calendar date
- **Primary key:** `date_key`, represented as an integer in `yyyyMMdd` format

### Output Columns
| Column | Description |
|---|---|
| `date_key` | Integer date key, such as `20260805` |
| `full_date` | Calendar date |
| `day` | Day of the month |
| `month` | Numeric month |
| `month_name` | Full month name |
| `quarter` | Calendar quarter |
| `year` | Calendar year |
| `day_of_week` | Full weekday name |
| `_gold_processed_timestamp` | Timestamp when the Gold dimension was processed |

### Data Quality Checks
- Confirm the date range is valid and contains transaction dates.
- Verify the generated calendar has the expected number of rows.
- Ensure `date_key` values are unique and non-null.
- Ensure `full_date` and `_gold_processed_timestamp` are non-null.
- Read the saved Parquet output back and verify its row count.

### Downstream Usage
The date dimension will be joined to `gold_fact_sales` using `date_key` derived from each transaction's `transaction_date`. This enables analysis by day, month, quarter, year, and day of week.

### Design Notes
- The calendar range is derived dynamically from the Silver transactions data rather than hardcoded.
- The dimension includes dates with no transactions to support continuous calendar reporting.
- The initial design uses a business date key and does not require a surrogate key.
- The Silver layer remains unchanged; the dimension is written to a separate Gold output path.

### Configuration / Paths

In [1]:
from google.colab import drive
drive.mount("/content/drive")

FILE_PATH = "/content/drive/MyDrive/pyspark-ecommerce-analytics-data"

SILVER_TRANSACTION_PATH = f"{FILE_PATH}/silver/transactions"
GOLD_DATE_PATH = f"{FILE_PATH}/gold/dim_date"

Mounted at /content/drive


### Spark Session

In [2]:
from pyspark.sql import SparkSession

from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("gold-dim-date")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


### Load Silver Transaction

In [3]:
df_transaction_dates = (
    spark.read
    .format("parquet")
    .load(SILVER_TRANSACTION_PATH)
    .select("transaction_date")
)

df_transaction_dates.printSchema()
df_transaction_dates.show(10, truncate=False)

root
 |-- transaction_date: date (nullable = true)

+----------------+
|transaction_date|
+----------------+
|2024-04-04      |
|2024-04-07      |
|2025-04-20      |
|2024-07-31      |
|2025-07-17      |
|2025-07-02      |
|2024-03-19      |
|2024-02-11      |
|2024-04-16      |
|2024-04-02      |
+----------------+
only showing top 10 rows


### Determine the Date Range

In [4]:
date_range = df_transaction_dates.agg(
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date")
).first()

# Store the boundaries for building the calendar.
MIN_TRANSACTION_DATE = date_range["min_date"]
MAX_TRANSACTION_DATE = date_range["max_date"]

# Fail clearly if the Silver dataset contains no usable transaction dates.
if MIN_TRANSACTION_DATE is None or MAX_TRANSACTION_DATE is None:
    raise ValueError(
        "Cannot build gold_dim_date: no valid transaction dates were found."
    )

print("Minimum transaction date:", MIN_TRANSACTION_DATE)
print("Maximum transaction date:", MAX_TRANSACTION_DATE)

Minimum transaction date: 2024-01-01
Maximum transaction date: 2025-12-31


### Generate the Calendar Dates

In [5]:
# Generate a continuous sequence of calendar dates.
df_date_base = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(MIN_TRANSACTION_DATE).cast("date"),
                F.lit(MAX_TRANSACTION_DATE).cast("date"),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("full_date")
    )
)

df_date_base.orderBy("full_date").show(10, truncate=False)

print("Generated calendar rows:", df_date_base.count())

+----------+
|full_date |
+----------+
|2024-01-01|
|2024-01-02|
|2024-01-03|
|2024-01-04|
|2024-01-05|
|2024-01-06|
|2024-01-07|
|2024-01-08|
|2024-01-09|
|2024-01-10|
+----------+
only showing top 10 rows
Generated calendar rows: 731


### Create Date Attributes

In [6]:
df_gold_dim_date = (
    df_date_base
    .select(
        # Integer key such as 20260805 for August 5, 2026.
        F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),

        # Keep the actual date for readable reporting and joins.
        F.col("full_date"),

        # Extract the numeric day, month, quarter, and year.
        F.dayofmonth("full_date").alias("day"),
        F.month("full_date").alias("month"),

        # Create a readable month name for dashboard labels.
        F.date_format("full_date", "MMMM").alias("month_name"),

        # Extract the calendar quarter, from 1 through 4.
        F.quarter("full_date").alias("quarter"),

        # Extract the calendar year.
        F.year("full_date").alias("year"),

        # Provide a readable day-of-week label.
        F.date_format("full_date", "EEEE").alias("day_of_week")
    )
)


df_gold_dim_date.printSchema()
df_gold_dim_date.orderBy("full_date").show(15, truncate=False)

root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = false)
 |-- day: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- month_name: string (nullable = false)
 |-- quarter: integer (nullable = false)
 |-- year: integer (nullable = false)
 |-- day_of_week: string (nullable = false)

+--------+----------+---+-----+----------+-------+----+-----------+
|date_key|full_date |day|month|month_name|quarter|year|day_of_week|
+--------+----------+---+-----+----------+-------+----+-----------+
|20240101|2024-01-01|1  |1    |January   |1      |2024|Monday     |
|20240102|2024-01-02|2  |1    |January   |1      |2024|Tuesday    |
|20240103|2024-01-03|3  |1    |January   |1      |2024|Wednesday  |
|20240104|2024-01-04|4  |1    |January   |1      |2024|Thursday   |
|20240105|2024-01-05|5  |1    |January   |1      |2024|Friday     |
|20240106|2024-01-06|6  |1    |January   |1      |2024|Saturday   |
|20240107|2024-01-07|7  |1    |January   |1      |2024|Sund

### Add Gold Audit Metadata

In [7]:
df_gold_dim_date = df_gold_dim_date.withColumn(
    "_gold_processed_timestamp",
    F.current_timestamp()
)

df_gold_dim_date.printSchema()

root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = false)
 |-- day: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- month_name: string (nullable = false)
 |-- quarter: integer (nullable = false)
 |-- year: integer (nullable = false)
 |-- day_of_week: string (nullable = false)
 |-- _gold_processed_timestamp: timestamp (nullable = false)



### Validate the Date Dimension

In [8]:
# Count rows and check for invalid keys, dates, and audit timestamps.
date_metrics = df_gold_dim_date.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("date_key").alias("distinct_date_keys"),
    F.sum(F.when(F.col("date_key").isNull(), 1).otherwise(0)).alias("null_date_keys"),
    F.sum(F.when(F.col("full_date").isNull(), 1).otherwise(0)).alias("null_full_dates"),
    F.sum(
        F.when(F.col("_gold_processed_timestamp").isNull(), 1).otherwise(0)
    ).alias("null_audit_timestamps")
).first()

# Calculate the expected number of dates, including both endpoints.
EXPECTED_DATE_COUNT = (MAX_TRANSACTION_DATE - MIN_TRANSACTION_DATE).days + 1

# Print the validation metrics.
print("Expected calendar rows:", EXPECTED_DATE_COUNT)
print("Actual calendar rows:", date_metrics["row_count"])
print("Distinct date keys:", date_metrics["distinct_date_keys"])
print("Null date keys:", date_metrics["null_date_keys"])
print("Null full dates:", date_metrics["null_full_dates"])
print("Null audit timestamps:", date_metrics["null_audit_timestamps"])

# Stop execution if any core validation fails.
assert date_metrics["row_count"] == EXPECTED_DATE_COUNT, "Unexpected calendar row count."
assert date_metrics["distinct_date_keys"] == date_metrics["row_count"], "Duplicate date keys found."
assert date_metrics["null_date_keys"] == 0, "Null date keys found."
assert date_metrics["null_full_dates"] == 0, "Null full dates found."
assert date_metrics["null_audit_timestamps"] == 0, "Null audit timestamps found."

print("Date dimension validation passed.")

Expected calendar rows: 731
Actual calendar rows: 731
Distinct date keys: 731
Null date keys: 0
Null full dates: 0
Null audit timestamps: 0
Date dimension validation passed.


### Write Gold Date Dimension

In [9]:
(
    df_gold_dim_date
    .write
    .format("parquet")
    .mode("overwrite")
    .save(GOLD_DATE_PATH)
)

print("Gold date dimension written to:", GOLD_DATE_PATH)

Gold date dimension written to: /content/drive/MyDrive/pyspark-ecommerce-analytics-data/gold/dim_date


### Read Back and Verify

In [10]:
df_gold_dim_date_check = (
    spark.read
    .format("parquet")
    .load(GOLD_DATE_PATH)
)

saved_date_count = df_gold_dim_date_check.count()

print("Expected saved rows:", EXPECTED_DATE_COUNT)
print("Actual saved rows:", saved_date_count)

df_gold_dim_date_check.printSchema()
df_gold_dim_date_check.orderBy("full_date").show(15, truncate=False)

assert saved_date_count == EXPECTED_DATE_COUNT, "Saved date dimension row count mismatch."

print("Gold date dimension read-back verification passed.")

Expected saved rows: 731
Actual saved rows: 731
root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- _gold_processed_timestamp: timestamp (nullable = true)

+--------+----------+---+-----+----------+-------+----+-----------+--------------------------+
|date_key|full_date |day|month|month_name|quarter|year|day_of_week|_gold_processed_timestamp |
+--------+----------+---+-----+----------+-------+----+-----------+--------------------------+
|20240101|2024-01-01|1  |1    |January   |1      |2024|Monday     |2026-10-29 15:35:31.999404|
|20240102|2024-01-02|2  |1    |January   |1      |2024|Tuesday    |2026-10-29 15:35:31.999404|
|20240103|2024-01-03|3  |1    |January   |1      |2024|Wednesday  |2026-10-29 15:35:31.999404|


### Quality Metrics and Run Summary

In [11]:
date_summary = {
    "table": "gold_dim_date",
    "min_transaction_date": str(MIN_TRANSACTION_DATE),
    "max_transaction_date": str(MAX_TRANSACTION_DATE),
    "expected_date_count": EXPECTED_DATE_COUNT,
    "saved_row_count": saved_date_count,
    "column_count": len(df_gold_dim_date_check.columns),
    "output_path": GOLD_DATE_PATH,
    "validation_status": "PASSED"
}

for metric, value in date_summary.items():
    print(f"{metric}: {value}")

table: gold_dim_date
min_transaction_date: 2024-01-01
max_transaction_date: 2025-12-31
expected_date_count: 731
saved_row_count: 731
column_count: 9
output_path: /content/drive/MyDrive/pyspark-ecommerce-analytics-data/gold/dim_date
validation_status: PASSED
